In [ ]:
from pathlib import Path
import os

import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from importlib import reload

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "core").exists() and (REPO_ROOT.parent / "core").exists():
    os.chdir(REPO_ROOT.parent)
    REPO_ROOT = Path.cwd()

from core import config, data, infer, net, pipelines, simulation, train
from helpers import utils

config.Config.configure_mt_amide_preclinical()

In [ ]:
mouse_name = 'ped_C1_1L_20241113_1245'

do_all_points = True  # set False to cut a few mins off run time (compare posteriors for a few points only)

In [ ]:
def get_lims(data_xa):
    xmin = list(np.nansum(data_xa['roi_mask_nans'][:,0,:].to_numpy(), axis=1) > 0).index(True) - 1
    xmax = 1 - list(np.nansum(data_xa['roi_mask_nans'][:,0,:].to_numpy(), axis=1) > 0)[::-1].index(True)
    ymin = list(np.nansum(data_xa['roi_mask_nans'][:,0,:].to_numpy(), axis=0) > 0).index(True) - 1
    ymax = 1 - list(np.nansum(data_xa['roi_mask_nans'][:,0,:].to_numpy(), axis=0) > 0)[::-1].index(True)
    return xmin, xmax, ymin, ymax

In [ ]:

data_xa = xr.open_dataset(f'data/{mouse_name}.nc' )
xmin, xmax, ymin, ymax = get_lims(data_xa)
data_xa_cutout = data_xa.isel(
    height=slice(xmin, xmax),
    width=slice(ymin, ymax) 
)

data_xa

In [ ]:
results_folder = f'./figs/mice/{mouse_name}'
# ensure results folder exists
os.makedirs(results_folder, exist_ok=True)

In [ ]:

plt.figure(figsize=(10, 2))
plt.subplot(121); plt.title('T1ms')
plt.imshow((data_xa_cutout['T1ms'].data * data_xa_cutout['roi_mask_nans']).squeeze(), vmin=800, vmax=2500)
plt.colorbar()
plt.subplot(122); plt.title('T2ms')
plt.imshow((data_xa_cutout['T2ms'].data * data_xa_cutout['roi_mask_nans']).squeeze(), vmin=25, vmax=120)
plt.colorbar()

data.SlicesFeed.norm_type = 'l2'
data_xa_cutout['B0_shift_ppm_map'] = (('height','slice','width'), np.zeros_like(data_xa_cutout['T1ms']))
data_xa_cutout['B1_fix_factor_map'] = (('height','slice','width'), np.ones_like(data_xa_cutout['T1ms']))

data_feed_mt = data.SlicesFeed.from_xarray(data_xa_cutout, seq_name='mt')
data_feed_amide = data.SlicesFeed.from_xarray(data_xa_cutout, seq_name='amide')

In [ ]:
# data_xa_cutout shoudl give unioform
show_B0_B1_maps = False
if show_B0_B1_maps:
    plt.figure(figsize=(10, 2))
    plt.subplot(121); plt.title('B0 shift (ppm)')
    plt.imshow((data_xa['roi_mask_nans'] * data_xa['B0_shift_ppm_map']).squeeze(), vmin=-0.5, vmax=0.5, cmap='bwr')
    plt.colorbar()
    plt.subplot(122); plt.title('B1 fix factor - 1')
    plt.imshow((data_xa['roi_mask_nans'] * data_xa['B1_fix_factor_map']).squeeze() - 1, cmap='bwr', vmin=-0.2, vmax=0.2)
    plt.colorbar()

In [ ]:
import time

t0 = time.time()
predictor_mt, predictor_amide = pipelines.run_train(
    datafeed_mt=data_feed_mt,
    ckpt_folder=os.path.abspath(f'./ckpts/mouse'),     
    do_cest=True, datafeed_cest=data_feed_amide 
) 
train_time = time.time() - t0
print(f"Total training time: {train_time/60:.1f} min")

In [ ]:
tumor_mask = data_xa_cutout['tumor_mask'].to_numpy().copy().squeeze()
contralateral_mask = data_xa_cutout['contralateral_mask'].to_numpy().copy().squeeze()

In [ ]:
data_feed_mt.ds = 1    
mt_tissue_params_est, MT_nn_pred_signal_normed_np = infer.infer(
    data_feed_mt, nn_predictor=predictor_mt, 
    do_forward=True, pool2predict='c'
    )
data_feed_amide.ds = 1    
amide_tissue_params_est, APT_nn_pred_signal_normed_np = infer.infer(
    data_feed_amide, nn_predictor=predictor_amide, 
    do_forward=True, pool2predict='b'
    )

In [ ]:
os.makedirs(f'{results_folder}/intermediates', exist_ok=True)

np.savez(
    f'{results_folder}/intermediates/mt_tissue_params_est.npz',
    MT_nn_pred_signal_normed_np=MT_nn_pred_signal_normed_np,
    tumor_mask=tumor_mask,
    contralateral_mask=contralateral_mask,
    **mt_tissue_params_est
)    
np.savez(
    f'{results_folder}/intermediates/amide_tissue_params_est.npz',    
    APT_nn_pred_signal_normed_np=APT_nn_pred_signal_normed_np,
    tumor_mask=tumor_mask,
    contralateral_mask=contralateral_mask,
    **amide_tissue_params_est
)    

In [ ]:
def plot_nn_pred_simple(data_feed, tissue_params_est, nn_pred_signal_normed_np, is_amide=False):
    plt.figure(figsize=(12, 2)) 
    plt.subplot(1,3,1); plt.title(("f$_{s}$" if is_amide else "f$_{ss}$") + " NN estimate", fontsize=10)    
    plt.imshow(100*tissue_params_est['fb_T' if is_amide else 'fc_T'].squeeze(), vmin=0, vmax=0.5 if is_amide else 30);               plt.colorbar()
    plt.subplot(1,3,2); plt.title("k$_{sw}$" if is_amide else "k$_{ssw}$" + " NN estimate", fontsize=10)
    plt.imshow(tissue_params_est['kb_T' if is_amide else 'kc_T'].squeeze(), vmin=0, vmax=600 if is_amide else 70, cmap='magma'); plt.colorbar()
    plt.subplot(1,3,3); plt.title("fitting error (NRMSE)", fontsize=10)
    nrmse_map = np.linalg.norm(nn_pred_signal_normed_np-data_feed.measured_normed_T, axis=0)/np.linalg.norm(data_feed.measured_normed_T, axis=0) 
    plt.imshow(nrmse_map.squeeze(), cmap='hot', vmin=0, vmax=0.06); plt.colorbar()
    
plot_nn_pred_simple(data_feed_mt, mt_tissue_params_est, MT_nn_pred_signal_normed_np)

In [ ]:

plt.figure(figsize=(6,3));
plt.subplot(121); plt.imshow(tumor_mask + contralateral_mask*2, cmap='gray')

plt.subplot(122); plt.imshow(mt_tissue_params_est['fc_T'].squeeze(), cmap='viridis', vmin = 0, vmax=0.2)  #  * (mask1 + mask2)
utils.remove_spines(plt.gca()); cbar=plt.colorbar(orientation="horizontal", pad=0.03); cbar.set_label(r'f$_{ss}$')
ax = plt.gca()
ax.contour(tumor_mask.astype(float), levels=[0.5], colors='orange', linewidths=1.5)
ax.contour(contralateral_mask.astype(float), levels=[0.5], colors='b', linewidths=1.5)

for format in ['svg', 'png']:
    plt.savefig(results_folder+f'/regions.{format}', bbox_inches='tight', dpi=300, format=format)


In [ ]:
# Calculate centroid of a binary mask
def get_mask_centroid(mask):
    coords = np.argwhere(mask == 1)        
    return np.int32(coords.mean(axis=0))

# Auto-create sample points as centroids
points_tumor = [get_mask_centroid(tumor_mask)]
points_contralateral = [get_mask_centroid(contralateral_mask)]

## MT UQ

### Visualization - human impression of uncertainty

In [ ]:
import analyze_uncertainty as au; reload(au)

f_est, k_est, cov_nnpred_scaled, f_sigma, k_sigma, height, width, angle = \
    au.extract_nn_estimate_of_posterior(mt_tissue_params_est, data_feed_mt.shape, is_cest=False)

In [ ]:
import matplotlib.patches as patches

reload(au)
sli = 0

axes = au.plot_CI_maps(
    f_est[:,sli,:], f_sigma[:,sli,:], 
    k_est[:,sli,:], k_sigma[:,sli,:], 
    is_cest=False, vmax_k=60, vmax_f=20, figsize=(11, 5), ktop=True
)

# ! manual defs of sample point been here
colors = ['blue','gold']
points = points_tumor + points_contralateral
labels = [1]*len(points_tumor) + [0]*len(points_contralateral)
for jj, (label, point) in enumerate(zip(labels, points)):
    square = patches.Rectangle((point[1]-.6, point[0]-.6), 1.5, 1.5, facecolor='none', edgecolor=colors[label], linewidth=2.5)
    axes[1, 1].add_patch(square)
    axes[1, 1].text(point[1]-(3 if label else -3), point[0]+1, ['A', 'B'][jj], color=colors[label], fontsize=15, ha='center')
    
for format in ['svg', 'png']:
    plt.savefig(results_folder+f'/MT_UQ_maps.{format}', bbox_inches='tight', dpi=300, format=format)

In [ ]:
#### removed content - alternative visualizations - see pre 2025-Nov25

### RoI-based aggregation


In [ ]:
# ACROSS THE MASKS!!
points_mask1 = np.argwhere(tumor_mask==1).tolist()
points_mask2 = np.argwhere(contralateral_mask==1).tolist()
all_masked_points = points_mask1 + points_mask2
all_masked_labels = [1]*len(points_mask1) + [0]*len(points_mask2)
ds = 1

#### APPLICATION I - physics-informed biomarker learning via posterior sampling

In [ ]:
reload(au)

au.demo_classification_app(
        f_est, k_est, cov_nnpred_scaled,
        all_masked_points, all_masked_labels, sli, 
        is_mt = True, result_fig_name=results_folder+f'/app1_physinf_clsf_MT.{format}'
    )


#### Sanity check - are the CRs similar to dist. of MAP over \[ $\approx$ homogeneous\] RoI? 

In [ ]:
f_est_sli = f_est[:,sli,:]
k_est_sli = k_est[:,sli,:]
x_coords, y_coords = zip(*points_mask1)
x_coords2, y_coords2 = zip(*points_mask2)
print(r"$\hat{f}_{ss}$ = "+f"{np.nanmean(f_est_sli[x_coords, y_coords]):.2f}$\\pm${np.nanstd(f_est_sli[x_coords, y_coords]):.2f}\\%  (tumor) " )
print(r"$\hat{f}_{ss}$ = "+f"{np.nanmean(f_est_sli[x_coords2, y_coords2]):.2f}$\\pm${np.nanstd(f_est_sli[x_coords2, y_coords2]):.2f}\\%  (contralateral) " )
print(r"$\hat{k}_{ssw}$ = "+f"{np.nanmean(k_est_sli[x_coords, y_coords]):.1f}$\\pm${np.nanstd(k_est_sli[x_coords, y_coords]):.1f}s$^{-1}$  (tumor) " )
print(r"$\hat{k}_{ssw}$ = "+f"{np.nanmean(k_est_sli[x_coords2, y_coords2]):.1f}$\\pm${np.nanstd(k_est_sli[x_coords2, y_coords2]):.1f}s$^{-1}$  (contralateral) " )
print(" ----- voxelwise posterior stds ----- ")
cov_nnpred_scaled_sli = cov_nnpred_scaled[:,sli,:,...]
f_pred_std = np.sqrt(cov_nnpred_scaled_sli[x_coords, y_coords, 0, 0])
k_pred_std = np.sqrt(cov_nnpred_scaled_sli[x_coords, y_coords, 1, 1])
f_pred_std2 = np.sqrt(cov_nnpred_scaled_sli[x_coords2, y_coords2, 0, 0])
k_pred_std2 = np.sqrt(cov_nnpred_scaled_sli[x_coords2, y_coords2, 1, 1])
print(r"Mean $\sigma_{f_{ss}}$ (tumor) = "+f"{np.nanmean(f_pred_std):.2f}$\\pm${np.nanstd(f_pred_std):.2f}\\%")
print(r"Mean $\sigma_{f_{ss}}$ (contralateral) = "+f"{np.nanmean(f_pred_std2):.2f}\\%")
print(r"Mean $\sigma_{k_{ssw}}$ (tumor) = "+f"{np.nanmean(k_pred_std):.1f}\\s$^{-1}$") 
print(r"Mean $\sigma_{k_{ssw}}$ (contralateral) = "+f"{np.nanmean(k_pred_std2):.1f}\\s$^{-1}$")

In [ ]:
from IPython.display import display, Latex
import pandas as pd

df = pd.DataFrame({  # \ estimate
    'RoI': ['Tumor', 'Contralateral'],
    '$\\hat{f}_{ss}$\\ (\\%)': [
        f"{np.nanmean(f_est_sli[x_coords, y_coords]):.1f}$\\pm${np.nanstd(f_est_sli[x_coords, y_coords]):.1f}", 
        f"{np.nanmean(f_est_sli[x_coords2, y_coords2]):.1f}$\\pm${np.nanstd(f_est_sli[x_coords2, y_coords2]):.1f}",
        ],
    'UQ:\\ $\\hat{\\sigma}(f_{ss})$\\ (\\%)': [
        f"{np.nanmean(f_pred_std):.2f}$\\pm${np.nanstd(f_pred_std):.2f}",
        f"{np.nanmean(f_pred_std2):.2f}$\\pm${np.nanstd(f_pred_std2):.2f}"
        ],
    '$\\hat{k}_{ssw}\\ (s^{-1}$)': [
        f"{np.nanmean(k_est_sli[x_coords, y_coords]):.1f}$\\pm${np.nanstd(k_est_sli[x_coords, y_coords]):.1f}", 
        f"{np.nanmean(k_est_sli[x_coords2, y_coords2]):.1f}$\\pm${np.nanstd(k_est_sli[x_coords2, y_coords2]):.1f}",
        ],
    'UQ:\\ $\\hat{\\sigma}(k_{ssw})\\ (s^{-1}$)': [
        f"{np.nanmean(k_pred_std):.2f}$\\pm${np.nanstd(k_pred_std):.2f}",
        f"{np.nanmean(k_pred_std2):.2f}$\\pm${np.nanstd(k_pred_std2):.2f}"
        ]    
}) 
# Generate LaTeX code (default is tabular)
latex_code = df.to_latex(index=False, escape=False, column_format='l|cc|r')
# Convert 'tabular' to 'array' for MathJax/KaTeX rendering in Notebooks
# 1. Replace environment name
latex_array = latex_code.replace('tabular', 'array')
# 2. Replace booktabs rules with hline (if present)
latex_array = latex_array.replace('$','')
latex_array = latex_array.replace('\\toprule', '\\hline').replace('\\midrule', '\\hline').replace('\\bottomrule', '\\hline')
# 3. Wrap in math mode delimiters $$ ... $$
latex_display = f"$$ \n{latex_array}\n $$"
display(Latex(latex_display))
print (latex_code)

### Explore UQ validation vs reference

#### Qualitative - display a single/dual-voxel drill-down

In [ ]:
reload(au)

# Sample tumor/contralateral points
_= au.compare_posteriors(
    points_contralateral+points_tumor, data_feed_mt, 
    mt_tissue_params_est, voxels2sample=8, folder_name=results_folder+'/MT_voxels'
)

In [ ]:
#  Drill down further into sampling-based quantification
# (demonstration of sampled-Mahalanobis quantification mechanism)
if False:
    au.single_voxel_detailed_viz(*points_contralateral[0], data_feed_mt, mt_tissue_params_est, sli=0)

#### Quantitative - compute mahalanobises over all voxels

In [ ]:
reload(au)

def meshgrid_xy_pairs(shape):
    x, y = np.meshgrid(np.arange(shape[0]), np.arange(shape[1]), indexing='ij')
    xy_pairs = np.stack([x, y], axis=-1).reshape(-1, 2)
    return xy_pairs.tolist()


In [ ]:
xy_points = meshgrid_xy_pairs(data_feed_mt.shape[0::2])
if not do_all_points:
    xy_points = xy_points[::10]

mahas1, mahas2, CR_areas, nnCR_areas, good_xy_points, \
    modes_of_exact_posterior, means_of_exact_posterior, covs_of_exact_posterior, timings, \
    exact_posterior_min_nrmse_percs, nn_posterior_min_nrmse_percs, extra_metrics = \
    au.compare_posteriors(
        xy_points, data_feed_mt, mt_tissue_params_est, 
        voxels2sample=1000, visualize=False
    )       

In [ ]:
os.makedirs(results_folder+'/intermediates', exist_ok=True)
np.savez_compressed(
    results_folder+'/intermediates/MT_UQ_stats_allpoints.npz',
    mahas1=mahas1, mahas2=mahas2,
    CR_areas=CR_areas,  nnCR_areas=nnCR_areas,
    good_xy_points=good_xy_points, timings=timings, train_time=train_time,
    f_est=f_est, k_est=k_est, cov_nnpred_scaled=cov_nnpred_scaled,
    covs_of_exact_posterior=covs_of_exact_posterior, modes_of_exact_posterior=modes_of_exact_posterior, means_of_exact_posterior=means_of_exact_posterior,    
    exact_posterior_min_nrmse_percs=exact_posterior_min_nrmse_percs, nn_posterior_min_nrmse_percs=nn_posterior_min_nrmse_percs,
    **extra_metrics
)
zz = np.load(
    results_folder+'/intermediates/MT_UQ_stats_allpoints.npz'
)
zz['mahas1'].shape

In [ ]:
np.mean(zz['exact_posterior_min_nrmse_percs']), np.mean(zz['nn_posterior_min_nrmse_percs']),\
np.std(zz['exact_posterior_min_nrmse_percs']), np.std(zz['nn_posterior_min_nrmse_percs'])

In [ ]:
#if False: # Load and extract variables from the npz file
if True: 
    npz_data = np.load(results_folder+'/intermediates/MT_UQ_stats_allpoints.npz', allow_pickle=True)
    mahas1 = npz_data['mahas1']
    mahas2 = npz_data['mahas2']
    CR_areas = npz_data['CR_areas']
    nnCR_areas = npz_data['nnCR_areas']
    good_xy_points = npz_data['good_xy_points']
    modes_of_exact_posterior = npz_data['modes_of_exact_posterior']
    means_of_exact_posterior = npz_data['means_of_exact_posterior']    
    covs_of_exact_posterior = npz_data['covs_of_exact_posterior']
    timings = npz_data['timings']
    train_time = npz_data['train_time']
    exact_posterior_min_nrmse_percs = npz_data['exact_posterior_min_nrmse_percs']
    nn_posterior_min_nrmse_percs = npz_data['nn_posterior_min_nrmse_percs']

In [ ]:
from IPython.display import Markdown, display

print('Sampled voxels: ', len(good_xy_points)), 
print("Before averaging voxels: ")
for mm in (4.5, 4, 3.5, 3):
    print(f"Mahas above {mm}: " +
        f"{100*sum(np.array(mahas1)>mm)/len(np.array(mahas1)):.1f}%, "+
        f"{100*sum(np.array(mahas2)>mm)/len(np.array(mahas2)):.1f}%"
    )
mean_mahas1 = np.reshape(mahas1, (len(good_xy_points), 10)).mean(axis=1)
mean_mahas2 = np.reshape(mahas2, (len(good_xy_points), 10)).mean(axis=1)
print("After averaging voxels: ")
for mm in (4.5, 4, 3.5, 3):
    print(f"Mahas above {mm}: " +
        f"{100*sum(np.array(mean_mahas1)>mm)/len(np.array(mean_mahas1)):.1f}%, "+
        f"{100*sum(np.array(mean_mahas2)>mm)/len(np.array(mean_mahas2)):.1f}%"
    )

mean_t = np.mean(timings)
std_t = np.std(timings)
latex_inline = rf"${mean_t:.2f}\pm{std_t:.2f}$"
print(latex_inline)
display(Markdown(f"Mean ± std (inline LaTeX): {latex_inline}"))
print(rf"whole slice: ${len(good_xy_points)*mean_t:.1f}\pm{np.sqrt(len(good_xy_points))*std_t:.1f} sec$")
print(rf"whole slice: ${len(good_xy_points)*mean_t/60:.1f}\pm{np.sqrt(len(good_xy_points))*std_t/60:.2f} mins$")


In [ ]:
heatmap = np.zeros(data_feed_mt.shape[0::2])
for (_x, _y), maha in zip(good_xy_points, mean_mahas1):
    heatmap[_x, _y] = maha
plt.figure(figsize=(12, 2));
plt.subplot(133)
plt.imshow(heatmap.squeeze(), vmin=0, vmax=5, cmap='hot'); plt.colorbar();
plt.title('Mean cross-mahalanobis 1 \n (NN samples to exact posterior) \n- is there a spatial pattern?', fontsize=10)
print(heatmap[4, 15:])
fss_dict_est_map, kss_dict_est_map = [np.zeros(data_feed_mt.shape[0::2]) for jj in range(2)]
for (_x, _y), f_k_dict_est in zip(good_xy_points, modes_of_exact_posterior):
    fss_dict_est_map[_x, _y], kss_dict_est_map[_x, _y] = f_k_dict_est  
plt.subplot(131)
plt.imshow(fss_dict_est_map.squeeze(), vmin=0, vmax=30, cmap='viridis'); plt.colorbar();
plt.title('f$_{ss}$ dict estimate', fontsize=10) 
plt.subplot(132)
plt.imshow(kss_dict_est_map.squeeze(), vmin=0, vmax=70, cmap='magma'); plt.colorbar();
plt.title('k$_{ssw}$ dict estimate', fontsize=10) \

if False:  # compare means to mods
    plt.figure(figsize=(12, 2));
    for (_x, _y), f_k_dict_est in zip(good_xy_points, means_of_exact_posterior):
        fss_dict_est_map[_x, _y], kss_dict_est_map[_x, _y] = f_k_dict_est  
    plt.subplot(131)
    plt.imshow(fss_dict_est_map.squeeze(), vmin=0, vmax=30, cmap='viridis'); plt.colorbar();
    plt.title('f$_{ss}$ dict estimate (post.mean)', fontsize=10) 
    plt.subplot(132)
    plt.imshow(kss_dict_est_map.squeeze(), vmin=0, vmax=70, cmap='magma'); plt.colorbar();
    plt.title('k$_{ssw}$ dict estimate (post.mean)', fontsize=10) \
    
plot_nn_pred_simple(data_feed_mt, mt_tissue_params_est, MT_nn_pred_signal_normed_np)

In [ ]:
au.summarize_mahas(mahas1, mahas2, good_xy_points) 
plt.xlabel('Mahalanobis Distance')

for format in ['svg', 'png']:
    plt.savefig(results_folder+f'/MT_mahas_hist.{format}', bbox_inches='tight', dpi=300, format=format)

#### Quantitative - univariate CI intersection

In [ ]:
f_best_dotprod_arr, k_best_dotprod_arr = zip(*means_of_exact_posterior)

au.analyze_CI_intersections(
    f_est, np.array(f_best_dotprod_arr), k_est, np.array(k_best_dotprod_arr), 
    cov_nnpred_scaled, np.array(covs_of_exact_posterior), good_xy_points, 
    sli=sli, is_mt=True, figsize=(10, 8), df=0.3, dk=0.2,
    result_figure_path=f'{results_folder}/MT_UQ_bland_altman_sample.svg'
)

In [ ]:
f_est_nn_l = [f_est[_x, sli, _y] for (_x, _y) in good_xy_points]
k_est_nn_l = [k_est[_x, sli, _y] for (_x, _y) in good_xy_points]
cov_nnpred_scaled_l = [cov_nnpred_scaled[_x, sli, _y] for (_x, _y) in good_xy_points]

au.analyze_scatter(
    f_est_nn_l, np.array(f_best_dotprod_arr), k_est_nn_l, np.array(k_best_dotprod_arr), 
    cov_nnpred_scaled_l, np.array(covs_of_exact_posterior), is_mt=True
)

## AMIDE UQ

In [ ]:
data_feed_amide.ds = 1    
amide_tissue_params_est, APT_nn_pred_signal_normed_np = infer.infer(
    data_feed_amide, nn_predictor=predictor_amide, 
    do_forward=True, pool2predict='b'
    )

plot_nn_pred_simple(data_feed_amide, amide_tissue_params_est, APT_nn_pred_signal_normed_np, is_amide=True)

In [ ]:
import analyze_uncertainty as au
reload(au)
sli=_z=0
f_est, k_est, cov_nnpred_scaled, f_sigma, k_sigma, height, width, angle = \
    au.extract_nn_estimate_of_posterior(amide_tissue_params_est, data_feed_amide.shape, is_cest=True)

axes = au.plot_CI_maps(
    f_est[:,sli,:], f_sigma[:,sli,:], 
    k_est[:,sli,:], k_sigma[:,sli,:], 
    is_cest=True, vmax_k=600, vmax_f=0.5, figsize=(11, 5), ktop=True
)
colors = ['blue','gold']
points = points_tumor + points_contralateral
labels = [1]*len(points_tumor) + [0]*len(points_contralateral)
for jj, (label, point) in enumerate(zip(labels, points)):
    square = patches.Rectangle((point[1]-.6, point[0]-.6), 1.5, 1.5, facecolor='none', edgecolor=colors[label], linewidth=2.5)
    axes[1, 1].add_patch(square)
    axes[1, 1].text(point[1]-(3 if label else -3), point[0]+1, ['A', 'B'][jj], color=colors[label], fontsize=15, ha='center')
    
for format in ['svg', 'png']:
    plt.savefig(results_folder+f'/APT_UQ_maps.{format}', bbox_inches='tight', dpi=300, format=format)   

In [ ]:
reload(au)

_ = au.compare_posteriors(
        points_contralateral+points_tumor, data_feed_mt, mt_tissue_params_est, 
        cest_tissue_params_est=amide_tissue_params_est, data_feed_cest=data_feed_amide, is_cest=True,
        mt_sim_mode='expm_bmmat', seq_name='amide', max_k_cest=600,
        voxels2sample=10, visualize=True, sli=0, folder_name=results_folder+'/APT_voxels'     
    )   

In [ ]:
reload(au)

au.demo_classification_app(
        f_est, k_est, cov_nnpred_scaled,
        all_masked_points, all_masked_labels, sli, 
        is_mt=False, result_fig_name=results_folder+f'/app1_physinf_clsf_APT'
    )

In [ ]:
f_est_sli = f_est[:,sli,:]
k_est_sli = k_est[:,sli,:]
x_coords, y_coords = zip(*points_mask1)
x_coords2, y_coords2 = zip(*points_mask2)
print(r"$\hat{f}_{s}$ = "+f"{np.nanmean(f_est_sli[x_coords, y_coords]):.2f}$\\pm${np.nanstd(f_est_sli[x_coords, y_coords]):.3f}\\%  (tumor) " )
print(r"$\hat{f}_{s}$ = "+f"{np.nanmean(f_est_sli[x_coords2, y_coords2]):.2f}$\\pm${np.nanstd(f_est_sli[x_coords2, y_coords2]):.3f}\\%  (contralateral) " )
print(r"$\hat{k}_{sw}$ = "+f"{np.nanmean(k_est_sli[x_coords, y_coords]):.1f}$\\pm${np.nanstd(k_est_sli[x_coords, y_coords]):.1f}s$^{-1}$  (tumor) " )
print(r"$\hat{k}_{sw}$ = "+f"{np.nanmean(k_est_sli[x_coords2, y_coords2]):.1f}$\\pm${np.nanstd(k_est_sli[x_coords2, y_coords2]):.1f}s$^{-1}$  (contralateral) " )
print(" ----- voxelwise posterior stds ----- ")
cov_nnpred_scaled_sli = cov_nnpred_scaled[:,sli,:,...]
f_pred_std = np.sqrt(cov_nnpred_scaled_sli[x_coords, y_coords, 0, 0])
k_pred_std = np.sqrt(cov_nnpred_scaled_sli[x_coords, y_coords, 1, 1])
f_pred_std2 = np.sqrt(cov_nnpred_scaled_sli[x_coords2, y_coords2, 0, 0])
k_pred_std2 = np.sqrt(cov_nnpred_scaled_sli[x_coords2, y_coords2, 1, 1])
print(r"Mean $\sigma_{f_{s}}$ (tumor) = "+f"{np.nanmean(f_pred_std):.3f}\\%")
print(r"Mean $\sigma_{f_{s}}$ (contralateral) = "+f"{np.nanmean(f_pred_std2):.3f}\\%")
print(r"Mean $\sigma_{k_{sw}}$ (tumor) = "+f"{np.nanmean(k_pred_std):.1f}\\s$^{-1}$") 
print(r"Mean $\sigma_{k_{sw}}$ (contralateral) = "+f"{np.nanmean(k_pred_std2):.1f}\\s$^{-1}$")

In [ ]:
from IPython.display import display, Latex
import pandas as pd

df = pd.DataFrame({  # \ estimate
    'RoI': ['Tumor', 'Contralateral'],
    '$\\hat{f}_{s}$\\ (\\%)': [
        f"{np.nanmean(f_est_sli[x_coords, y_coords]):.3f}$\\pm${np.nanstd(f_est_sli[x_coords, y_coords]):.3f}", 
        f"{np.nanmean(f_est_sli[x_coords2, y_coords2]):.3f}$\\pm${np.nanstd(f_est_sli[x_coords2, y_coords2]):.3f}",
        ],
    'UQ:\\ $\\hat{\\sigma}(f_{s}$)\\ (\\%)': [
        f"{np.nanmean(f_pred_std):.3f}$\\pm${np.nanstd(f_pred_std):.3f}",
        f"{np.nanmean(f_pred_std2):.3f}$\\pm${np.nanstd(f_pred_std2):.3f}"
        ],
    '$\\hat{k}_{sw}\\ (s^{-1})$': [
        f"{np.nanmean(k_est_sli[x_coords, y_coords]):.1f}$\\pm${np.nanstd(k_est_sli[x_coords, y_coords]):.1f}", 
        f"{np.nanmean(k_est_sli[x_coords2, y_coords2]):.1f}$\\pm${np.nanstd(k_est_sli[x_coords2, y_coords2]):.1f}",
        ],
    'UQ:\\ $\\hat{\\sigma}(k_{sw})\\ (s^{-1})$': [
        f"{np.nanmean(k_pred_std):.1f}$\\pm${np.nanstd(k_pred_std):.1f}",
        f"{np.nanmean(k_pred_std2):.1f}$\\pm${np.nanstd(k_pred_std2):.1f}"
        ]    
}) 
# Generate LaTeX code (default is tabular)
latex_code = df.to_latex(index=False, escape=False, column_format='l|cc|r')
# Convert 'tabular' to 'array' for MathJax/KaTeX rendering in Notebooks
# 1. Replace environment name
latex_array = latex_code.replace('tabular', 'array')
# 2. Replace booktabs rules with hline (if present)
latex_array = latex_array.replace('\\toprule', '\\hline').replace('\\midrule', '\\hline').replace('\\bottomrule', '\\hline')
latex_array = latex_array.replace('$','')
# 3. Wrap in math mode delimiters $$ ... $$
latex_display = f"$$ \n{latex_array}\n $$"
display(Latex(latex_display))
print(latex_code)

#### Compare all voxels

In [ ]:
xy_points = meshgrid_xy_pairs(data_feed_mt.shape[0::2])

do_all_points = True
if not do_all_points:
    xy_points = xy_points[::8]
    
mahas1, mahas2, CR_areas, nnCR_areas, good_xy_points,\
    modes_of_exact_posterior, means_of_exact_posterior, covs_of_exact_posterior, timings, \
    exact_posterior_min_nrmse_percs, nn_posterior_min_nrmse_percs, extra_metrics = \
    au.compare_posteriors(
        xy_points, data_feed_mt, mt_tissue_params_est, 
        cest_tissue_params_est=amide_tissue_params_est, data_feed_cest=data_feed_amide, is_cest=True, seq_name='amide', 
        voxels2sample=1000, visualize=False
    )   

In [ ]:
from IPython.display import Markdown, display

print('Sampled voxels: ', len(good_xy_points)), 
print("Before averaging voxels: ")
for mm in (4.5, 4, 3.5, 3):
    print(f"Mahas above {mm}: " +
        f"{100*sum(np.array(mahas1)>mm)/len(np.array(mahas1)):.1f}%, "+
        f"{100*sum(np.array(mahas2)>mm)/len(np.array(mahas2)):.1f}%"
    )
mean_mahas1 = np.reshape(mahas1, (len(good_xy_points), 10)).mean(axis=1)
mean_mahas2 = np.reshape(mahas2, (len(good_xy_points), 10)).mean(axis=1)
print("After averaging voxels: ")
for mm in (4.5, 4, 3.5, 3):
    print(f"Mahas above {mm}: " +
        f"{100*sum(np.array(mean_mahas1)>mm)/len(np.array(mean_mahas1)):.1f}%, "+
        f"{100*sum(np.array(mean_mahas2)>mm)/len(np.array(mean_mahas2)):.1f}%"
    )    
mean_t = np.mean(timings)
std_t = np.std(timings)
latex_inline = rf"${mean_t:.2f}\pm{std_t:.2f}$"
print(latex_inline)
display(Markdown(f"Mean ± std (inline LaTeX): {latex_inline}"))
print(rf"whole slice: ${len(good_xy_points)*mean_t:.1f}\pm{np.sqrt(len(good_xy_points))*std_t:.1f} sec$")
print(rf"whole slice: ${len(good_xy_points)*mean_t/60:.1f}\pm{np.sqrt(len(good_xy_points))*std_t/60:.2f} mins$")


In [ ]:
os.makedirs(results_folder+'/intermediates', exist_ok=True)
np.savez_compressed(
    results_folder+'/intermediates/APT_UQ_stats_allpoints.npz',
    mahas1=mahas1, mahas2=mahas2,
    CR_areas=CR_areas,  nnCR_areas=nnCR_areas,
    good_xy_points=good_xy_points, 
    f_est=f_est, k_est=k_est, cov_nnpred_scaled=cov_nnpred_scaled,
    covs_of_exact_posterior=covs_of_exact_posterior, modes_of_exact_posterior=modes_of_exact_posterior, means_of_exact_posterior=means_of_exact_posterior,
    timings=timings, exact_posterior_min_nrmse_percs=exact_posterior_min_nrmse_percs, nn_posterior_min_nrmse_percs=nn_posterior_min_nrmse_percs,
    **extra_metrics
)
zz = np.load(
    results_folder+'/intermediates/APT_UQ_stats_allpoints.npz'
)
zz['mahas1'].shape

In [ ]:
if False: # Load and extract variables from the npz file
    npz_data = np.load(results_folder+'/intermediates/APT_UQ_stats_allpoints.npz', allow_pickle=True)
    mahas1 = npz_data['mahas1']
    mahas2 = npz_data['mahas2']
    CR_areas = npz_data['CR_areas']
    nnCR_areas = npz_data['nnCR_areas']
    good_xy_points = npz_data['good_xy_points']
    modes_of_exact_posterior = npz_data['modes_of_exact_posterior']
    means_of_exact_posterior = npz_data['means_of_exact_posterior']
    covs_of_exact_posterior = npz_data['covs_of_exact_posterior']
    timings = npz_data['timings']
    exact_posterior_min_nrmse_percs = npz_data['exact_posterior_min_nrmse_percs']
    nn_posterior_min_nrmse_percs = npz_data['nn_posterior_min_nrmse_percs']

In [ ]:
reload(au)

au.summarize_mahas(mahas1, mahas2, good_xy_points) 
plt.xlabel('Mahalanobis Distance')
for format in ['svg', 'png']:       
    plt.savefig(results_folder+f'/APT_mahas_hist.{format}', bbox_inches='tight', dpi=300, format=format)

In [ ]:
reload(au)
f_best_dotprod_arr, k_best_dotprod_arr = zip(*means_of_exact_posterior)

au.analyze_CI_intersections(
    f_est, np.array(f_best_dotprod_arr), k_est, np.array(k_best_dotprod_arr), 
    cov_nnpred_scaled, np.array(covs_of_exact_posterior), good_xy_points, 
    sli=sli, is_mt=False, figsize=(10, 8), df=0.008, dk=15, fmin=0.07, fmax=0.3,
    result_figure_path=f'{results_folder}/APT_UQ_bland_altman_sample.svg'
)

In [ ]:
reload(au)

f_est_nn_l = [f_est[_x, sli, _y] for (_x, _y) in good_xy_points]
k_est_nn_l = [k_est[_x, sli, _y] for (_x, _y) in good_xy_points]
cov_nnpred_scaled_l = [cov_nnpred_scaled[_x, sli, _y] for (_x, _y) in good_xy_points]

au.analyze_scatter(
    f_est_nn_l, np.array(f_best_dotprod_arr), k_est_nn_l, np.array(k_best_dotprod_arr), 
    cov_nnpred_scaled_l, np.array(covs_of_exact_posterior), is_mt=False
)

In [ ]:
_mahas1 = np.reshape(mahas1, (len(good_xy_points), 10)).mean(axis=1)
heatmap = np.zeros(data_feed_mt.shape[0::2])
for (_x, _y), maha in zip(good_xy_points, _mahas1):
    heatmap[_x, _y] = maha
plt.figure(figsize=(12, 2));
plt.subplot(133)
plt.imshow(heatmap.squeeze(), vmin=0, vmax=5, cmap='hot'); plt.colorbar();
plt.title('Mean cross-mahalanobis 1 \n (NN samples to exact posterior) \n- is there a spatial pattern?', fontsize=10)
print(heatmap[4, 15:])
fs_dict_est_map, ksw_dict_est_map = [np.zeros(data_feed_mt.shape[0::2]) for jj in range(2)]
for (_x, _y), f_k_dict_est in zip(good_xy_points, modes_of_exact_posterior):
    fs_dict_est_map[_x, _y], ksw_dict_est_map[_x, _y] = f_k_dict_est  
plt.subplot(131)
plt.imshow(fs_dict_est_map.squeeze(), vmin=0, vmax=0.5, cmap='viridis'); plt.colorbar();
plt.title('f$_{s}$ dict estimate', fontsize=10) 
plt.subplot(132)
plt.imshow(ksw_dict_est_map.squeeze(), vmin=0, vmax=600, cmap='magma'); plt.colorbar();
plt.title('k$_{sw}$ dict estimate', fontsize=10) 

plot_nn_pred_simple(data_feed_amide, amide_tissue_params_est, APT_nn_pred_signal_normed_np, is_amide=True)

## APPLICATION II (Figs. 9-10) - 

### Tracking shrinking posteriors for offline protocol optimization and online adaptation (Fig. 10)

Computing Posteriors at partial (subsequence) acquisitions - first steps towards sequence optimality analysis, and online adaptativity / early stopping

In [ ]:
os.makedirs(results_folder+'/app2shrink', exist_ok=True)

#### Single-voxel CRs

In [ ]:
import analyze_uncertainty as au
# from matplotlib import gridspec
reload(au); reload(au.apps)

_x, _y = points_contralateral[0] 
_x += 1
_signal = data_feed_mt.measured_normed_T[:, _x, 0, _y][:,None,None]
n_meas = _signal.shape[0]
loc_dict_res = 200

f_best_dotprod, k_best_dotprod, dict_signal, nrmse, _df, _dk = au.get_nrmse_grid(
    None, _x, _y, 0, 
    data_feed_mt=data_feed_mt,
    data_feed_cest=None, seq_name='mt', 
    loc_dict_res=loc_dict_res
)            

## (!!)
cumsum_sqerr, pdf = au.get_progressive_pdf(_signal, dict_signal)

sample_voxel_CRs_indices, f_prog_est, k_prog_est, CR_areas, f_CIs, k_CIs = \
    au.apps.plot_progressive_pdfs(cumsum_sqerr, pdf, n_meas, _df, _dk, loc_dict_res, figsize=(20, 4), sample_voxel_CRs_indices = list(range(2, n_meas+1, 4)))

for format in ['svg', 'png']:
    plt.savefig(results_folder+f'/app2shrink/sample_{_x}_{_y}.svg', bbox_inches='tight', dpi=300, format='svg')



In [ ]:
# Repeat without skips to get the full sequence of quantified metrics for this voxel
sample_voxel_CRs_indices, f_prog_est, k_prog_est, CR_areas, f_CIs, k_CIs = \
    au.apps.plot_progressive_pdfs(cumsum_sqerr, pdf, n_meas, _df, _dk, loc_dict_res, figsize=(10, 0.2), sample_voxel_CRs_indices=list(range(3, 30)))
sample_voxel_CRs_indices = np.array(sample_voxel_CRs_indices)
CR_areas = np.array(CR_areas)

#### Repeat across many random voxels, collect all trends

In [ ]:
reload(au)

voxels2sample = 100  # ~ 1/sec
sli=0
_xy_points, all_voxels_CRs_indices, CR_area_trends, f_CI_trends, k_CI_trends, f_err_trends, k_err_trends, f_vals, k_vals = \
    au.plot_CRs_decrease_along_sequence(
        data_feed_mt, sli=sli, seq_name='mt', 
        voxels2sample=voxels2sample,        
    )

all_voxels_CRs_indices, f_err_trends, k_err_trends, f_CI_trends, k_CI_trends = \
    np.array(all_voxels_CRs_indices), np.array(f_err_trends), np.array(k_err_trends), np.array(f_CI_trends), np.array(k_CI_trends)


In [ ]:
import matplotlib as mpl
from mpl_toolkits.axes_grid1 import make_axes_locatable

def plot_bounds(f_CI_trends, k_CI_trends, f_err_trends, k_err_trends, aspect_equal=True, fig=None, vert=False):
    fig = fig or plt.figure(figsize=(6,6) if vert else (7,3), constrained_layout=True)
    ax = fig.add_subplot(2,1,1) if vert else fig.add_subplot(1,2,1)
    xx, yy = f_CI_trends.flatten()/2, np.abs(f_err_trends.flatten())
    scat = ax.scatter(
        xx, yy,
        c = np.tile(np.arange(f_CI_trends.shape[1]), f_CI_trends.shape[0]), cmap='RdYlGn',
        marker='o', alpha=0.15
    )
    print(f"{100*np.sum(xx>yy)/len(xx):.1f}% of points have f error < CI/2")
    mappable = mpl.cm.ScalarMappable(norm=scat.norm, cmap=plt.get_cmap("RdYlGn"))
    if vert:
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="10%", pad=0.05)
        cbar = fig.colorbar(mappable, ax=ax, cax=cax)
        cbar.set_label('# scans acquired')
    if aspect_equal:
        ax.set_aspect('equal', adjustable='box')
    ax.set_xlim(0., 10); ax.set_ylim(0, 10)
    ax.plot([.1, 20], [.1, 20], 'k--', linewidth=0.5)
    ax.set_xlabel('$\hat{f}_{ss}$ CI/2 (%)')
    ax.set_ylabel('|$\Delta\hat{f}_{ss}|$ (%)')    
    ax = fig.add_subplot(2,1,2) if vert else fig.add_subplot(1,2,2)
    xx, yy = k_CI_trends.flatten()/2, np.abs(k_err_trends.flatten())
    scat = ax.scatter(
        xx, yy, 
        c = np.tile(np.arange(k_CI_trends.shape[1]), k_CI_trends.shape[0]), cmap='RdYlGn',
        marker='o', alpha=0.1
    )
    print(f"{100*np.sum(xx>yy)/len(xx):.1f}% of points have k error < CI/2")
    mappable = mpl.cm.ScalarMappable(norm=scat.norm, cmap=plt.get_cmap("RdYlGn"))
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="10%", pad=0.05 )
    cbar = fig.colorbar(mappable, ax=ax, cax=cax)
    cbar.set_label('Number of scans acquired')
    if aspect_equal:
        ax.set_aspect('equal', adjustable='box')
    ax.set_xlim(0.1, 35); ax.set_ylim(0.1, 35)
    ax.plot([1.5, 40], [1.5, 40], 'k--', linewidth=0.5)
    ax.set_xlabel('$\hat{k}_{ssw}$ CI/2 (s$^{-1}$)')
    ax.set_ylabel('|$\Delta\hat{k}_{ssw}|$ (s$^{-1}$)')

plot_bounds(f_CI_trends, k_CI_trends, f_err_trends, k_err_trends)
for format in ['svg', 'png']:
    plt.savefig(results_folder+f'/app2shrink/evolving_CIvsErr.{format}', bbox_inches='tight', dpi=300, format=format)


In [ ]:
import matplotlib as mpl

mpl.rcParams['font.size'] = 12
mpl.rcParams['axes.labelsize'] = 12
mpl.rcParams['xtick.labelsize'] = 12

In [ ]:
sample_voxel_CRs_indices, all_voxels_CRs_indices

In [ ]:
reload(au.apps); 

au.apps.plot_CR_area_vs_measurements(
    all_voxels_CRs_indices, sample_voxel_CRs_indices, 
    CR_areas, CR_area_trends, f_CI_trends, k_CI_trends, 
    data_feed_mt
)

for fmt in ['svg', 'png']:
    plt.savefig(results_folder+f'/app2shrink/CRCItrends_protocol.{fmt}', bbox_inches='tight', dpi=300, format=fmt)


### Rapid dictionary-based quantification of whole slice across subsequences (w.o. posterior)

Instead of computing "local dictionaries" for individual voxels as above, let's take a conventional (multi-dimensional global) dictionary, and match partials of the measured signal to partials of the simulated ones. This will give us progressive estimates, so we can compute their pseudo-errors (taking the very last estimate as reference). Of course, no free lunch, so no progressive CRs/CIs this way. We can however take these error as "ground truth" discrepancy between "mid-sequence" and "final" estimates, and then see if CRs/CIs at mid-sequence correlate and provide a bound for the eventual diff, interpreted as "update of estimate". If yes, then these mid-sequence CRs/CIs are directly useful for decision support on stopping acquisition - if the expected update is small. Alternatively and more simply, one can look if CIs converged and stopped shrinking


In [ ]:
import collections
from dictionary_methods import dictbased_runner, dict_matcher

dict_as_slicefeed = dictbased_runner.get_dict(
    use_cartesian=True,
    parameter_values_od=collections.OrderedDict({
        'T1a_ms': np.arange(700, 3000+100, 100),
        'T2a_ms': np.concatenate((np.arange(20, 150, 10), np.arange(200, 1000+100, 100))),
        'B0_shift_ppm_map': [0.], 'B1_fix_factor_map': [1.], 'fb_gt_T': [0.], 'kb_gt_T': [0.],
        'fc_gt_T': np.arange(0.01/2, 0.3+.01/2, 0.01/2),
        'kc_gt_T': np.arange(2, 50+2, 2)
    }),     
    shape=(99, 80, 100)
)

In [ ]:
import scipy

def MAPE_vs_CI(f_CI_trends, k_CI_trends, f_MAPEs, k_MAPEs, MAE_nmeas_list, fig=None, fontsize=14):
    fig = fig or plt.figure(figsize=(10, 2))        
    ax = fig.add_subplot(1,1,1)
    r1, r2 = scipy.stats.pearsonr(np.median(f_CI_trends, axis=0), f_MAPEs), \
        scipy.stats.pearsonr(np.median(k_CI_trends, axis=0), k_MAPEs)
    print('Pearson r (f): ', r1)
    print('Pearson r (k): ', r2)
    ax.plot(MAE_nmeas_list, f_MAPEs, 'gv-', markersize=5, markerfacecolor='none', linewidth=1, label=r'$\hat{f}_{ss}$[n] error')  #r'$\mathbb{E}{|\hat{f}_{ss}[n]/\hat{f}_{ss}[N]-1|}$ (MAPE)')
    ax.plot(MAE_nmeas_list, k_MAPEs, 'g^-', markersize=5, markerfacecolor='none', linewidth=1, label=r'$\hat{k}_{ssw}$[n] error') #r'$\mathbb{E}{|\hat{k}_{ssw}[n]/\hat{k}_{ssw}[N]-1|}$ (MAPE)')
    ax.set_ylabel('MAPE (%)', fontsize=fontsize)
    ax.set_xlabel('Number of scans acquired (n)', fontsize=fontsize)
    ax.legend(fontsize=fontsize)
    ax.set_xlim(2, 32)    
    ax.grid()



## FINAL FIGURE (Fig. 9)

In [ ]:
import analyze_uncertainty as au
import analyze_uncertainty.apps as au_apps
import analyze_uncertainty.viz as au_viz
import matplotlib as mpl
import scipy
from mpl_toolkits.axes_grid1 import make_axes_locatable
from importlib import reload

reload(au_apps)
reload(au_viz)
reload(au)

base_fontsize = 16
mpl.rcParams.update({
    'font.size': base_fontsize,
    'axes.titlesize': base_fontsize + 1,
    'axes.labelsize': base_fontsize,
    'xtick.labelsize': base_fontsize - 1,
    'ytick.labelsize': base_fontsize - 1,
    'legend.fontsize': base_fontsize - 1,
    'figure.titlesize': base_fontsize + 2,
})

def slap_label_on_container(container, label_text, loc=(-0.01, 1.05), color='black', fontsize=base_fontsize + 4):
    ghost_ax = container.add_axes([0, 0, 1, 1], frameon=False)
    ghost_ax.set_xticks([])
    ghost_ax.set_yticks([])
    ghost_ax.text(
        loc[0],
        loc[1],
        label_text,
        transform=ghost_ax.transAxes,
        fontsize=fontsize,
        fontweight='bold',
        va='top',
        ha='left',
        color=color,
    )

def plot_bounds(f_CI_trends, k_CI_trends, f_err_trends, k_err_trends, aspect_equal=True, fig=None, vert=False, fontsize=base_fontsize):
    fig = fig or plt.figure(figsize=(6, 6) if vert else (7, 3), constrained_layout=True)
    ax = fig.add_subplot(2, 1, 1) if vert else fig.add_subplot(1, 2, 1)
    xx, yy = f_CI_trends.flatten() / 2, np.abs(f_err_trends.flatten())
    scat = ax.scatter(
        xx,
        yy,
        c=np.tile(np.arange(f_CI_trends.shape[1]), f_CI_trends.shape[0]),
        cmap='RdYlGn',
        marker='o',
        alpha=0.15,
    )
    print(f"{100 * np.sum(xx > yy) / len(xx):.1f}% of points have f error < CI/2")
    mappable = mpl.cm.ScalarMappable(norm=scat.norm, cmap=plt.get_cmap('RdYlGn'))
    if vert:
        divider = make_axes_locatable(ax)
        cax = divider.append_axes('right', size='10%', pad=0.05)
        cbar = fig.colorbar(mappable, ax=ax, cax=cax)
        cbar.set_label('# scans acquired', fontsize=fontsize - 1)
    if aspect_equal:
        ax.set_aspect('equal', adjustable='box')
    ax.set_xlim(0.0, 10)
    ax.set_ylim(0, 10)
    ax.plot([0.1, 20], [0.1, 20], 'k--', linewidth=0.5)
    ax.set_xlabel(r'$\hat{f}_{ss}$ CI/2 (%)', fontsize=fontsize)
    ax.set_ylabel(r'|$\Delta\hat{f}_{ss}$| (%)', fontsize=fontsize)
    ax.tick_params(labelsize=fontsize - 1)

    ax = fig.add_subplot(2, 1, 2) if vert else fig.add_subplot(1, 2, 2)
    xx, yy = k_CI_trends.flatten() / 2, np.abs(k_err_trends.flatten())
    scat = ax.scatter(
        xx,
        yy,
        c=np.tile(np.arange(k_CI_trends.shape[1]), k_CI_trends.shape[0]),
        cmap='RdYlGn',
        marker='o',
        alpha=0.1,
    )
    print(f"{100 * np.sum(xx > yy) / len(xx):.1f}% of points have k error < CI/2")
    mappable = mpl.cm.ScalarMappable(norm=scat.norm, cmap=plt.get_cmap('RdYlGn'))
    divider = make_axes_locatable(ax)
    cax = divider.append_axes('right', size='10%', pad=0.05)
    cbar = fig.colorbar(mappable, ax=ax, cax=cax)
    cbar.set_label('Number of scans acquired', fontsize=fontsize - 1)
    if aspect_equal:
        ax.set_aspect('equal', adjustable='box')
    ax.set_xlim(0.1, 35)
    ax.set_ylim(0.1, 35)
    ax.plot([1.5, 40], [1.5, 40], 'k--', linewidth=0.5)
    ax.set_xlabel(r'$\hat{k}_{ssw}$ CI/2 (s$^{-1}$)', fontsize=fontsize)
    ax.set_ylabel(r'|$\Delta\hat{k}_{ssw}$| (s$^{-1}$)', fontsize=fontsize)
    ax.tick_params(labelsize=fontsize - 1)

def MAPE_vs_CI(f_CI_trends, k_CI_trends, f_MAPEs, k_MAPEs, MAE_nmeas_list, fig=None, fontsize=base_fontsize):
    fig = fig or plt.figure(figsize=(10, 2))
    ax = fig.add_subplot(1, 1, 1)
    r1, r2 = scipy.stats.pearsonr(np.median(f_CI_trends, axis=0), f_MAPEs), scipy.stats.pearsonr(np.median(k_CI_trends, axis=0), k_MAPEs)
    print('Pearson r (f): ', r1)
    print('Pearson r (k): ', r2)
    ax.plot(MAE_nmeas_list, f_MAPEs, 'gv-', markersize=5, markerfacecolor='none', linewidth=1, label=r'$\hat{f}_{ss}$[n] error')
    ax.plot(MAE_nmeas_list, k_MAPEs, 'g^-', markersize=5, markerfacecolor='none', linewidth=1, label=r'$\hat{k}_{ssw}$[n] error')
    ax.set_ylabel('MAPE (%)', fontsize=fontsize)
    ax.set_xlabel('Number of scans acquired (n)', fontsize=fontsize)
    ax.legend(fontsize=fontsize - 1)
    ax.set_xlim(2, 32)
    ax.grid()
    ax.tick_params(labelsize=fontsize - 1)

fig = plt.figure(figsize=(21, 12.5), constrained_layout=True)
subfig_rows = fig.subfigures(3, 1, height_ratios=[4, 4, 4], hspace=0.05)
sfig_a, sfig_b = subfig_rows[0], subfig_rows[1]
sfig_cd, sfig_e, sfig_f = subfig_rows[2].subfigures(1, 3, width_ratios=[1, 0.8, 1.2], wspace=0.05)

subfigs_by_letter = {'a': sfig_a, 'b': sfig_b, 'c': sfig_cd, 'e': sfig_e, 'f': sfig_f}
for letter, subfig in subfigs_by_letter.items():
    slap_label_on_container(subfig, f'({letter})')
slap_label_on_container(sfig_cd, '(d)', loc=(-0.01, 0.38), color='black')

f_best, k_best, best_match, err_3d_dict = dictbased_runner.match_to_dict(dict_as_slicefeed, data_feed_mt, constrain_T1T2=False)
MAE_nmeas_list = list(range(3, n_meas + 1, 1))

f_MAPEs, k_MAPEs, f_MAEs, k_MAEs = au.apps.plot_progressive_maps_calc_MAPEs(
    dict_as_slicefeed,
    data_feed_mt,
    f_best,
    k_best,
    MAE_nmeas_list,
    fig=sfig_a,
    xy=points_contralateral[0],
)
_ = au.apps.plot_progressive_pdfs(
    cumsum_sqerr,
    pdf,
    n_meas,
    _df,
    _dk,
    loc_dict_res,
    sample_voxel_CRs_indices=list(range(2, n_meas + 1, 4)),
    fig=sfig_b,
)
au.apps.plot_CR_area_vs_measurements(
    all_voxels_CRs_indices,
    sample_voxel_CRs_indices,
    CR_areas,
    CR_area_trends,
    f_CI_trends,
    k_CI_trends,
    data_feed_mt,
    fig=sfig_cd,
 )
for axis in getattr(sfig_cd, 'axes', []):
    label_text = axis.get_ylabel()
    if label_text == '% of parameter range':
        axis.yaxis.label.set_size(base_fontsize - 2)
    elif label_text == r'$\\Delta\\omega/\\omega_0$ (ppm)':
        axis.yaxis.label.set_size(base_fontsize - 2)
MAPE_vs_CI(f_CI_trends, k_CI_trends, f_MAPEs, k_MAPEs, MAE_nmeas_list, fig=sfig_e)
plot_bounds(f_CI_trends, k_CI_trends, f_err_trends, k_err_trends, fig=sfig_f, aspect_equal=False, vert=False)

for format in ['svg', 'png']:
    plt.savefig(results_folder + f'/mouse_seq.{format}', bbox_inches='tight', dpi=300, format=format)
plt.savefig(results_folder + '/mouse_seq_LOWRES.png', bbox_inches='tight', dpi=100)

In [ ]:
results_folder

### Removed content - see older versions

- Residuals research (see before 2025-Dec-03)
- Convergence movie creation (see old movie.py backups)